# Olist E-Commerce Customer Intelligence

## Day 1 — Data Understanding

### Project Overview

This project analyzes the Olist Brazilian E-Commerce Public Dataset to understand
sales performance, customer behavior, product performance, delivery experience,
and customer reviews.

The project will use Python, SQL, RFM customer segmentation, and Power BI to
develop data-driven business insights and recommendations.

### Objective of Day 1

The purpose of this notebook is to understand the structure and quality of the
raw Olist dataset before performing any data cleaning or analysis.

In this notebook, we will:

- Load all 9 Olist datasets.
- Inspect the number of rows and columns.
- Examine columns and sample records.
- Inspect data types.
- Analyze missing values.
- Check duplicate rows.
- Examine important identifiers and their uniqueness.
- Understand the grain of each table.
- Identify potential primary and foreign keys.
- Document relationships between tables.
- Record initial data-quality observations.
- Identify questions that need to be addressed during data cleaning.

> **Important:** The raw datasets will not be modified in this notebook.
> Dataset-specific findings will be based on the actual Olist data.

In [29]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [30]:
DATA_PATH = "../data/raw/"

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

In [31]:
data = {}

for name, file in files.items():
    data[name] = pd.read_csv(os.path.join(DATA_PATH, file))

print(f"Loaded {len(data)} tables: {list(data.keys())}")

Loaded 9 tables: ['customers', 'geolocation', 'order_items', 'payments', 'reviews', 'orders', 'products', 'sellers', 'category_translation']


## Table Inventory
Rows, columns, missing cells, and fully duplicated rows for each table.

In [32]:
table_inventory = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    }
    for name, df in data.items()
])

table_inventory

,table,rows,columns,missing_cells,duplicate_rows,memory_mb
0,customers,99441,5,0,0,11.03
1,geolocation,1000163,5,0,261831,50.12
2,order_items,112650,7,0,0,18.37
3,payments,103886,5,0,0,8.11
4,reviews,99224,7,145903,0,17.84
5,orders,99441,8,4908,0,21.95
6,products,32951,9,2448,0,3.73
7,sellers,3095,4,0,0,0.22
8,category_translation,71,2,0,0,0.00


## Column Profile per Table
For every table: data type, non-null count, missing count and %, and number of unique values.

In [33]:
def profile_table(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })

for name, df in data.items():
    print(f"\n{'=' * 60}\n{name.upper()} — {df.shape[0]:,} rows × {df.shape[1]} columns\n{'=' * 60}")
    display(profile_table(df))


CUSTOMERS — 99,441 rows × 5 columns


,dtype,non_null,missing,missing_%,unique
customer_id,str,99441,0,0.0,99441
customer_unique_id,str,99441,0,0.0,96096
customer_zip_code_prefix,int64,99441,0,0.0,14994
customer_city,str,99441,0,0.0,4119
customer_state,str,99441,0,0.0,27



GEOLOCATION — 1,000,163 rows × 5 columns


,dtype,non_null,missing,missing_%,unique
geolocation_zip_code_prefix,int64,1000163,0,0.0,19015
geolocation_lat,float64,1000163,0,0.0,717360
geolocation_lng,float64,1000163,0,0.0,717613
geolocation_city,str,1000163,0,0.0,8011
geolocation_state,str,1000163,0,0.0,27



ORDER_ITEMS — 112,650 rows × 7 columns


,dtype,non_null,missing,missing_%,unique
order_id,str,112650,0,0.0,98666
order_item_id,int64,112650,0,0.0,21
product_id,str,112650,0,0.0,32951
seller_id,str,112650,0,0.0,3095
shipping_limit_date,str,112650,0,0.0,93318
price,float64,112650,0,0.0,5968
freight_value,float64,112650,0,0.0,6999


... (8 outputs left out)


SELLERS — 3,095 rows × 4 columns


,dtype,non_null,missing,missing_%,unique
seller_id,str,3095,0,0.0,3095
seller_zip_code_prefix,int64,3095,0,0.0,2246
seller_city,str,3095,0,0.0,611
seller_state,str,3095,0,0.0,23



CATEGORY_TRANSLATION — 71 rows × 2 columns


,dtype,non_null,missing,missing_%,unique
product_category_name,str,71,0,0.0,71
product_category_name_english,str,71,0,0.0,71


## Sample Records
First three rows of each table.

In [34]:
for name, df in data.items():
    print(f"\n{name.upper()}")
    display(df.head(3))


CUSTOMERS


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



GEOLOCATION


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



ORDER_ITEMS


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


... (8 outputs left out)


SELLERS


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



CATEGORY_TRANSLATION


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## Key and Relationship Checks

In [35]:
key_checks = {
    "customers.customer_id": data["customers"]["customer_id"].is_unique,
    "orders.order_id": data["orders"]["order_id"].is_unique,
    "products.product_id": data["products"]["product_id"].is_unique,
    "sellers.seller_id": data["sellers"]["seller_id"].is_unique,
    "order_items (order_id, order_item_id)": not data["order_items"].duplicated(["order_id", "order_item_id"]).any(),
    "payments (order_id, payment_sequential)": not data["payments"].duplicated(["order_id", "payment_sequential"]).any(),
    "reviews.review_id": data["reviews"]["review_id"].is_unique,
}
pd.Series(key_checks, name="is_unique")

customers.customer_id                       True
orders.order_id                             True
products.product_id                         True
sellers.seller_id                           True
order_items (order_id, order_item_id)       True
payments (order_id, payment_sequential)     True
reviews.review_id                          False
Name: is_unique, dtype: bool

In [36]:
orders_ids = set(data["orders"]["order_id"])

coverage = pd.DataFrame({
    "orders_without_items": [len(orders_ids - set(data["order_items"]["order_id"]))],
    "orders_without_payment": [len(orders_ids - set(data["payments"]["order_id"]))],
    "orders_without_review": [len(orders_ids - set(data["reviews"]["order_id"]))],
    "categories_without_translation": [len(
        set(data["products"]["product_category_name"].dropna())
        - set(data["category_translation"]["product_category_name"])
    )],
})
coverage.T.rename(columns={0: "count"})

,count
orders_without_items,775
orders_without_payment,1
orders_without_review,768
categories_without_translation,2


In [37]:
orders_per_customer = data["customers"].groupby("customer_unique_id")["customer_id"].nunique()
orders_per_customer.value_counts().sort_index()

customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [38]:
orders = data["orders"]
date_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
orders[date_cols].isna().groupby(orders["order_status"]).sum()

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [39]:
display(orders["order_status"].value_counts())
display(data["payments"]["payment_type"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [40]:
no_items = orders_ids - set(data["order_items"]["order_id"])
orders[orders["order_id"].isin(no_items)]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

## Findings Summary

### 1. Table Structure and Grain

| Table | Rows | One row represents | Key |
|---|---|---|---|
| customers | 99,441 | One customer_id (created per order) | customer_id |
| orders | 99,441 | One order | order_id |
| order_items | 112,650 | One item within an order | (order_id, order_item_id) |
| payments | 103,886 | One payment record of an order | (order_id, payment_sequential) |
| reviews | 99,224 | One review record | No unique key (review_id repeats) |
| products | 32,951 | One product | product_id |
| sellers | 3,095 | One seller | seller_id |
| geolocation | 1,000,163 | One coordinate record for a ZIP prefix | None (ZIP prefix repeats) |
| category_translation | 71 | One category name mapping | product_category_name |

### 2. Keys and Relationships
- All primary key candidates are unique except `reviews.review_id` (see Key Checks).
- 775 orders have no items, and they cannot contribute to product or revenue analysis. Status breakdown: [fill in].
- 1 order has no payment record.
- 768 orders have no review.
- 2 product categories have no English translation.
- Payments has 103,886 rows for 99,440 orders, so some orders are paid in multiple records. Payments must be aggregated to order level before joining with order_items to avoid double-counting revenue.

### 3. Missing Values
- Missing values occur only in `reviews`, `orders` and `products`.
- **reviews:** comment title is 88.34% missing and comment message 58.70% missing. These are optional fields, so this is expected rather than a data error.
- **orders:** missing approval (160), carrier (1,783) and customer delivery (2,965) dates are mostly explained by status. For example, all 1,107 shipped orders lack a delivery date because they had not yet arrived.
- **Anomalies:**
  - 8 delivered orders have no customer delivery date.
  - 14 delivered orders have no approval date.
  - 2 delivered orders have no carrier date.
  - 6 canceled orders have a customer delivery date (625 canceled vs 619 missing).
- **products:** 610 products are missing category, name length, description length and photo count together. 2 products are missing all physical dimensions.

### 4. Duplicates
- `geolocation` has 261,831 fully duplicated rows (26.2%), and ZIP prefixes repeat with slightly different coordinates. It must be aggregated to one row per ZIP prefix before joining.
- `reviews`: 814 rows share a review_id with another row, and 551 extra rows come from orders with more than one review.
- No other table has fully duplicated rows.

### 5. Data Types
- All date and timestamp columns are stored as text (`str`) in `orders`, `order_items` and `reviews`, and need datetime conversion.
- Column names `product_name_lenght` and `product_description_lenght` are misspelled in the source.

### 6. customer_id vs customer_unique_id
- 99,441 customer_id values map to 96,096 customer_unique_id values. A new customer_id is created for each order.
- 93,099 customers have one customer_id, and only 2,997 (3.1%) have more than one. The maximum is 17.
- **Decision:** customer-level, repeat-purchase and RFM analysis will use `customer_unique_id`.
- Because repeat customers are rare, Frequency will have little variation in RFM, and the scoring method must account for this.

### 7. Categorical Values
- **Order status:** 96,478 of 99,441 orders (97.0%) are delivered. The rest are shipped (1,107), canceled (625), unavailable (609), invoiced (314), processing (301), created (5) and approved (2).
- **Payment type:** credit card is the most common method (76,795 rows), followed by boleto (19,784), voucher (5,775) and debit card (1,529). 3 rows are `not_defined`.
- Customers come from 27 states, and sellers from 23.

### 8. Questions for Day 2 (Cleaning)
1. Which order statuses count toward revenue and customer analysis? Delivered only, or also shipped and invoiced?
2. Which revenue definition to use: item price + freight (order_items) or payment value (payments)?
3. How to treat the 8 delivered orders without a delivery date and the 6 canceled orders with one?
4. How to handle orders with multiple reviews and repeated review_ids?
5. How to aggregate geolocation to one row per ZIP prefix (e.g. median lat/lng)?
6. How to label the 610 products with no category and the 2 untranslated categories?
7. What to do with the 3 `not_defined` payment rows?

### 9. Questions for EDA
1. Are there sparse months at the start or end of the date range that distort trends?
2. How is delivery time related to review score?
3. How concentrated is revenue across categories, states and sellers?